# SVTRv2X: Enhanced Scene Text Recognition via Self-Distilled Mixture-of-Experts
### Google Colab Free Tier Guide (Working Datasets + Resuming Across Session Disconnects)
Paper: Guo et al., PLOS ONE 2026

This notebook provides:
1. **Instant Demo Dataset Generator**: Test training and evaluation immediately without downloading gigabytes.
2. **Official Public Dataset Mirrors**: Working Hugging Face and Cloud download links.
3. **Zero-Data-Loss Training**: Automatically save checkpoints to Google Drive and resume seamlessly when Colab disconnects.

## 1. Enable GPU
In Colab top menu: **Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU**.

In [ ]:
!nvidia-smi

## 2. Mount Google Drive (Essential to Save Progress Across Disconnects)
This ensures that checkpoints (`latest.pth`, `best.pth`) are saved directly to your permanent Google Drive.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Create permanent checkpoint directories on your Drive
os.makedirs('/content/drive/MyDrive/svtrv2x_checkpoints/phase1', exist_ok=True)
os.makedirs('/content/drive/MyDrive/svtrv2x_checkpoints/phase2', exist_ok=True)

## 3. Clone / Setup the Codebase
Clone your repository or extract `OpenOCR_v2X.zip`.

In [ ]:
# Option A: If uploaded as zip to Google Drive
# !unzip -q /content/drive/MyDrive/OpenOCR_v2X.zip -d /content/

# Option B: If cloned from your GitHub
# !git clone https://github.com/<your-username>/OpenOCR_v2X.git /content/OpenOCR_v2X

%cd /content/OpenOCR_v2X

## 4. Install Dependencies

In [ ]:
!pip install -q imgaug lmdb pyclipper pyyaml rapidfuzz tqdm huggingface_hub

## 5. Dataset Options

### Option A: Instant Demo Dataset (Recommended to start immediately!)
Generates all 8 LMDB benchmark splits (`document`, `scene`, `web`, `handwriting`) in 3 seconds directly on Colab so you can verify the full training pipeline right now:

In [ ]:
!python tools/create_demo_dataset.py

### Option B: Download Official Public STR & Union14M Benchmarks from Hugging Face
The authors maintain public benchmarks on Hugging Face (`topdu/OpenOCR-Data`).

In [ ]:
# Method 1: Using Python API (Recommended - Works on all versions):
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="topdu/OpenOCR-Data",
    repo_type="dataset",
    allow_patterns="evaluation/**",
    local_dir="/content/OpenOCR_v2X/"
)

# Method 2: Using the modern 'hf' CLI:
# !hf download topdu/OpenOCR-Data --include "evaluation/**" --repo-type dataset --local-dir /content/OpenOCR_v2X/

## 6. Run GPU Verification Test
Verifies Jumble Module, Top-1 MoE, SDM alignment, and backward pass on the Colab T4 GPU.

In [ ]:
!python tools/test_svtrv2x.py

## 7. Phase 1 Training (with Auto-Save to Google Drive)
On Colab Free Tier:
- Checkpoints are written to `/content/drive/MyDrive/svtrv2x_checkpoints/phase1`
- Checkpoints are saved every **1 epoch** (`save_epoch_step="[0, 1]"`)
- Batch size is set to `64` to fit inside Colab's 16GB T4 GPU VRAM

In [ ]:
!python tools/train_rec.py --c configs/rec/svtrv2/svtrv2x_ch_phase1.yml \
    -o Global.output_dir=/content/drive/MyDrive/svtrv2x_checkpoints/phase1 \
       Global.save_epoch_step="[0, 1]" \
       Train.sampler.first_bs=64 \
       Train.loader.batch_size_per_card=64

## 8. How to Resume Training When Colab Session Disconnects
Whenever Colab disconnects or reaches its session limit:
1. Open Colab and re-mount Google Drive (Step 2).
2. Run the command below. It reads `latest.pth` and automatically restores the model weights, optimizer, learning rate, and picks up at the exact next epoch!

In [ ]:
# RESUME COMMAND:
!python tools/train_rec.py --c configs/rec/svtrv2/svtrv2x_ch_phase1.yml \
    -o Global.output_dir=/content/drive/MyDrive/svtrv2x_checkpoints/phase1 \
       Global.checkpoints=/content/drive/MyDrive/svtrv2x_checkpoints/phase1/latest.pth \
       Global.save_epoch_step="[0, 1]" \
       Train.sampler.first_bs=64 \
       Train.loader.batch_size_per_card=64

## 9. Phase 2: Fine-Tuning with Self-Distillation (SDM) for 20 Epochs
Loads the Phase 1 checkpoint and fine-tunes for 20 epochs with SDM active (`lambda_3 = 0.5`).

In [ ]:
!python tools/train_rec.py --c configs/rec/svtrv2/svtrv2x_ch_phase2.yml \
    -o Global.pretrained_model=/content/drive/MyDrive/svtrv2x_checkpoints/phase1/best.pth \
       Global.output_dir=/content/drive/MyDrive/svtrv2x_checkpoints/phase2 \
       Global.save_epoch_step="[0, 1]" \
       Train.sampler.first_bs=64 \
       Train.loader.batch_size_per_card=64

## 10. Evaluation

In [ ]:
!python tools/eval_rec.py --c configs/rec/svtrv2/svtrv2x_ch_phase2.yml \
    -o Global.checkpoints=/content/drive/MyDrive/svtrv2x_checkpoints/phase2/best.pth